In [1]:
!pip install streamlit
!pip install pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.9 MB/s eta 0:00:00


In [3]:
NGROK_AUTH_TOKEN = "36T8q53H7RM7g6HH3WZQ5AP1Mrv_5M5AgvXEtKN9CymZfWCs3"
!ngrok config add-authtoken $NGROK_AUTH_TOKEN


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf

model = tf.keras.models.load_model(
    "/content/drive/MyDrive/unet_task4_final.h5",
    compile=False
)

print("MODEL LOADED SUCCESSFULLY!")


Mounted at /content/drive
MODEL LOADED SUCCESSFULLY!


In [5]:
%%writefile app_task6.py
import streamlit as st
import numpy as np
import tensorflow as tf
import cv2
from PIL import Image
import io

IMG_SIZE = 128
MODEL_PATH = "/content/drive/MyDrive/unet_task4_final.h5"

@st.cache_resource
def load_unet_model():
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    return model

def preprocess(pil_img):
    img = pil_img.convert("RGB")
    img_resized = img.resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img_resized) / 255.0
    return arr, img_resized

def predict_mask(model, img_array):
    pred = model.predict(np.expand_dims(img_array, 0))[0, :, :, 0]
    return (pred > 0.5).astype(np.uint8) * 255

model = load_unet_model()
st.title("Object Segmentation using UNet (Task-6)")
st.write("Upload an image → Model segments → Download mask")

uploaded_file = st.file_uploader("Upload Image", type=["jpg","jpeg","png"])

if uploaded_file:
    pil_img = Image.open(uploaded_file)
    arr_img, resized = preprocess(pil_img)

    st.subheader("Input Image (Resized)")
    st.image(resized)

    if st.button("Predict"):
        mask = predict_mask(model, arr_img)

        st.subheader("Predicted Mask")
        st.image(mask, clamp=True)

        buf = io.BytesIO()
        Image.fromarray(mask).save(buf, format="PNG")

        st.download_button("Download Mask",
                           data=buf.getvalue(),
                           file_name="pred_mask.png",
                           mime="image/png")


Writing app_task6.py


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("PUBLIC URL:", public_url)

!streamlit run app_task6.py --server.port 8501 --server.address 0.0.0.0


PUBLIC URL: NgrokTunnel: "https://polygalaceous-superwrought-willetta.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

2025-12-06 11:19:16.210892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765019956.295895    3397 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765019956.323684    3397 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765019956.384143    3397 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765019956.384203    3397 computation_placer.cc:17